In [24]:
from google.colab import drive
drive.mount('/gdrive')
current_dir = "/gdrive/MyDrive/First Challenge"
%cd $current_dir

Drive already mounted at /gdrive; to attempt to forcibly remount, call drive.mount("/gdrive", force_remount=True).
/gdrive/MyDrive/First Challenge


In [25]:
# Set seed for reproducibility
SEED = 399

# Import necessary libraries
import os

# Set environment variables before importing modules
os.environ['PYTHONHASHSEED'] = str(SEED)
os.environ['MPLCONFIGDIR'] = os.getcwd() + '/configs/'

# Suppress warnings
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=Warning)

# Import necessary modules
import logging
import random
import numpy as np

# Set seeds for random number generators in NumPy and Python
np.random.seed(SEED)
random.seed(SEED)

# Import PyTorch
import torch
torch.manual_seed(SEED)
from torch import nn
# from torchsummary import summary
from torch.utils.tensorboard import SummaryWriter
from torch.utils.data import TensorDataset, DataLoader
logs_dir = "tensorboard"
!pkill -f tensorboard
%load_ext tensorboard
!mkdir -p models

if torch.cuda.is_available():
    device = torch.device("cuda")
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = True
else:
    device = torch.device("cpu")

print(f"PyTorch version: {torch.__version__}")
print(f"Device: {device}")

# Import other libraries
import copy
import shutil
from itertools import product
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, confusion_matrix
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Configure plot display settings
sns.set(font_scale=1.4)
sns.set_style('white')
plt.rc('font', size=14)
%matplotlib inline

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard
PyTorch version: 2.9.0+cu126
Device: cuda


In [26]:
## Import libraries
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from tensorflow.keras.utils import to_categorical

## Load data
X_train = pd.read_csv('pirate_pain_train.csv')
y_train = pd.read_csv('pirate_pain_train_labels.csv')
X_test = pd.read_csv('pirate_pain_test.csv')


# optimize the running steps - from float64 to float32
joint_columns = [col for col in X_train.columns if col.startswith('joint_')]
survey_columns = [col1 for col1 in X_train.columns if col1.startswith('pain_survey_')]

# Convert the data type
for col in joint_columns:
    X_train[col] = X_train[col].astype('float32')
for col1 in survey_columns:
    X_train[col1] = X_train[col1].astype('float32')

X_train.info()


# Merge labels with training features
train = X_train.merge(y_train, on='sample_index')

# Mapping for legs, hands, eyes
mapping_legs = {'two': 2, 'one+peg_leg': 1}
mapping_hands = {'two': 2, 'one+hook_hand': 1}
mapping_eyes = {'two': 2, 'one+eye_patch': 1}

# Apply the mapping to training and test sets
for col, mapping in zip(['n_legs', 'n_hands', 'n_eyes'], [mapping_legs, mapping_hands, mapping_eyes]):
    train[col] = train[col].map(mapping)
    X_test[col] = X_test[col].map(mapping)


# Identify feature columns (exclude sample_index, time, and label)
feature_cols = [col for col in train.columns if col not in ['sample_index', 'time', 'label']]

# Group by sample_index to create sequences (samples, time_steps, features)
num_samples = train['sample_index'].nunique()
time_steps = train.groupby('sample_index').size().max()  # usually 180
num_features = len(feature_cols)

# Initialize array for features
X_array = np.zeros((num_samples, time_steps, num_features))

# Fill the array for each sample
for i, sample_id in enumerate(train['sample_index'].unique()):
    # Sort by time to keep sequence order
    sample_data = train[train['sample_index'] == sample_id].sort_values('time')[feature_cols].values
    X_array[i, :sample_data.shape[0], :] = sample_data

# Extract labels (one per sample)
y_labels = train.groupby('sample_index')['label'].first().values

# Encode labels to integers
encoder = LabelEncoder()
y_encoded = encoder.fit_transform(y_labels)

# One-hot encode for neural network
y_array = to_categorical(y_encoded, num_classes=3)

# Scale features (standardization)
X_flat = X_array.reshape(-1, num_features)  # flatten for scaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_flat)
X_array = X_scaled.reshape(num_samples, time_steps, num_features)

# X_array = ready input features (samples, time_steps, features)
# y_array = ready targets (one-hot encoded)
print("Feature array shape:", X_array.shape)
print("Target array shape:", y_array.shape)


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 105760 entries, 0 to 105759
Data columns (total 40 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   sample_index   105760 non-null  int64  
 1   time           105760 non-null  int64  
 2   pain_survey_1  105760 non-null  float32
 3   pain_survey_2  105760 non-null  float32
 4   pain_survey_3  105760 non-null  float32
 5   pain_survey_4  105760 non-null  float32
 6   n_legs         105760 non-null  object 
 7   n_hands        105760 non-null  object 
 8   n_eyes         105760 non-null  object 
 9   joint_00       105760 non-null  float32
 10  joint_01       105760 non-null  float32
 11  joint_02       105760 non-null  float32
 12  joint_03       105760 non-null  float32
 13  joint_04       105760 non-null  float32
 14  joint_05       105760 non-null  float32
 15  joint_06       105760 non-null  float32
 16  joint_07       105760 non-null  float32
 17  joint_08       105760 non-nul

In [27]:
# Check Data Distribution (for labels)
y_train['label'].value_counts(normalize=True)


,proportion
label,
no_pain,0.773071
low_pain,0.142209
high_pain,0.084720


In [28]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

# Convert labels to encoded integers (if you haven’t already)
y_encoded = encoder.transform(y_train['label'])

# Compute class weights
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_encoded),
    y=y_encoded
)

# Convert to dictionary format
class_weights = dict(enumerate(class_weights))
print("Class Weights:", class_weights)


Class Weights: {0: np.float64(3.9345238095238093), 1: np.float64(2.3439716312056738), 2: np.float64(0.43118069145466403)}


In [29]:
# Spliting data
from sklearn.model_selection import train_test_split

# stratify=y_encoded to maintain class balance in both sets
X_train_split, X_val_split, y_train_split, y_val_split = train_test_split(
    X_array, y_array,
    test_size=0.2,          # 20% validation
    random_state=42,
    stratify=encoder.transform(y_train['label'])
)

print("Training set:", X_train_split.shape, y_train_split.shape)
print("Validation set:", X_val_split.shape, y_val_split.shape)


Training set: (528, 160, 38) (528, 3)
Validation set: (133, 160, 38) (133, 3)


In [31]:
# ======================================================
#  Prepare Test Set
# ======================================================

# Apply same scaler to test data
num_features = len(feature_cols)
num_samples_test = X_test['sample_index'].nunique()
time_steps = train.groupby('sample_index').size().max()

# Fill test array
X_test_array = np.zeros((num_samples_test, time_steps, num_features))

for i, sample_id in enumerate(X_test['sample_index'].unique()):
    sample_data = X_test[X_test['sample_index'] == sample_id].sort_values('time')[feature_cols].values
    X_test_array[i, :sample_data.shape[0], :] = sample_data

# Scale using the previously fitted scaler
X_test_flat = X_test_array.reshape(-1, num_features)
X_test_scaled = scaler.transform(X_test_flat)
X_test_array = X_test_scaled.reshape(num_samples_test, time_steps, num_features)

print("Test array shape:", X_test_array.shape)

# ======================================================
# Train/Validation Split
# ======================================================
from sklearn.model_selection import train_test_split

y_encoded = np.argmax(y_array, axis=1)  # convert one-hot to class indices
X_train_split, X_val_split, y_train_split, y_val_split = train_test_split(
    X_array, y_encoded, test_size=0.2, random_state=SEED, stratify=y_encoded
)

print("Train shape:", X_train_split.shape)
print("Validation shape:", X_val_split.shape)

# ======================================================
# Convert to PyTorch Tensors and Dataloaders
# ======================================================
import torch
from torch.utils.data import TensorDataset, DataLoader

X_train_tensor = torch.tensor(X_train_split, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train_split, dtype=torch.long)
X_val_tensor = torch.tensor(X_val_split, dtype=torch.float32)
y_val_tensor = torch.tensor(y_val_split, dtype=torch.long)

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
val_dataset = TensorDataset(X_val_tensor, y_val_tensor)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

# ======================================================
# Define Model
# ======================================================
import torch.nn as nn
import torch.nn.functional as F

class CNN1D(nn.Module):
    def __init__(self, num_features, num_classes=3):
        super(CNN1D, self).__init__()
        self.conv1 = nn.Conv1d(num_features, 64, kernel_size=3)
        self.conv2 = nn.Conv1d(64, 64, kernel_size=3)
        self.dropout = nn.Dropout(0.5)
        self.fc = nn.Linear(64, num_classes)

    def forward(self, x):
        x = x.permute(0, 2, 1)  # (batch, features, time)
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = F.adaptive_avg_pool1d(x, 1).squeeze(-1)
        x = self.dropout(x)
        x = self.fc(x)
        return x

model = CNN1D(num_features=num_features).to(device)
print(model)

# ======================================================
# Training Setup
# ======================================================
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss(
    weight=torch.tensor(list(class_weights.values()), dtype=torch.float32).to(device)
)

# Early stopping parameters
best_f1 = 0
patience = 5
wait = 0
best_model_path = "models/best_model.pth"

# ======================================================
# Training Loop with Early Stopping
# ======================================================
from sklearn.metrics import f1_score

for epoch in range(30):
    # ---- Training ----
    model.train()
    train_loss, train_preds, train_labels = 0, [], []
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
        train_preds.extend(outputs.argmax(1).cpu().numpy())
        train_labels.extend(y_batch.cpu().numpy())

    train_f1 = f1_score(train_labels, train_preds, average='macro')

    # ---- Validation ----
    model.eval()
    val_preds, val_labels = [], []
    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            outputs = model(X_batch)
            val_preds.extend(outputs.argmax(1).cpu().numpy())
            val_labels.extend(y_batch.cpu().numpy())

    val_f1 = f1_score(val_labels, val_preds, average='macro')

    print(f"Epoch {epoch+1:02d} | Train F1: {train_f1:.4f} | Val F1: {val_f1:.4f}")

    # ---- Early Stopping ----
    if val_f1 > best_f1:
        best_f1 = val_f1
        wait = 0
        torch.save(model.state_dict(), best_model_path)
    else:
        wait += 1
        if wait >= patience:
            print("Early stopping triggered!")
            break

# ======================================================
# Load Best Model and Evaluate on Validation Set
# ======================================================
model.load_state_dict(torch.load(best_model_path))
model.eval()

val_preds, val_labels = [], []
with torch.no_grad():
    for X_batch, y_batch in val_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        outputs = model(X_batch)
        val_preds.extend(outputs.argmax(1).cpu().numpy())
        val_labels.extend(y_batch.cpu().numpy())

print("Final Validation F1:", f1_score(val_labels, val_preds, average='macro'))

# ======================================================
# Generate Test Predictions
# ======================================================
X_test_tensor = torch.tensor(X_test_array, dtype=torch.float32).to(device)
model.eval()
test_preds = []
with torch.no_grad():
    outputs = model(X_test_tensor)
    test_preds = outputs.argmax(1).cpu().numpy()

# Decode class labels back to original strings
test_labels = encoder.inverse_transform(test_preds)

# Prepare submission
submission = pd.DataFrame({
    "sample_index": X_test['sample_index'].unique(),
    "label": test_labels
})

submission.to_csv("submission_01.csv", index=False)
print("Submission file saved as submission.csv")


Test array shape: (1324, 160, 38)
Train shape: (528, 160, 38)
Validation shape: (133, 160, 38)
CNN1D(
  (conv1): Conv1d(38, 64, kernel_size=(3,), stride=(1,))
  (conv2): Conv1d(64, 64, kernel_size=(3,), stride=(1,))
  (dropout): Dropout(p=0.5, inplace=False)
  (fc): Linear(in_features=64, out_features=3, bias=True)
)
Epoch 01 | Train F1: 0.2296 | Val F1: 0.3691
Epoch 02 | Train F1: 0.4991 | Val F1: 0.5492
Epoch 03 | Train F1: 0.5821 | Val F1: 0.6228
Epoch 04 | Train F1: 0.6469 | Val F1: 0.6757
Epoch 05 | Train F1: 0.6642 | Val F1: 0.7180
Epoch 06 | Train F1: 0.6448 | Val F1: 0.7425
Epoch 07 | Train F1: 0.7086 | Val F1: 0.7964
Epoch 08 | Train F1: 0.7620 | Val F1: 0.7318
Epoch 09 | Train F1: 0.7234 | Val F1: 0.7894
Epoch 10 | Train F1: 0.7486 | Val F1: 0.8006
Epoch 11 | Train F1: 0.7935 | Val F1: 0.7951
Epoch 12 | Train F1: 0.7793 | Val F1: 0.8750
Epoch 13 | Train F1: 0.7967 | Val F1: 0.8570
Epoch 14 | Train F1: 0.8165 | Val F1: 0.8787
Epoch 15 | Train F1: 0.8211 | Val F1: 0.8539
Epoch 